# Xu Xiang: image generator

Reads `images.csv` (built by `build_shotlist.py`) and makes every drama still
for the film with **FLUX.1-schnell** (Apache 2.0) from Hugging Face, on
Kaggle's free T4 GPUs.

It makes `VARIANTS` versions of each image so you can pick the best, and
writes a contact sheet per image plus `review.html` to look through them all.

## One-time setup

1. **Hugging Face.** Sign in, open
   [black-forest-labs/FLUX.1-schnell](https://huggingface.co/black-forest-labs/FLUX.1-schnell)
   and accept the licence (the model is free to use commercially, but the page
   asks you to agree first). Then make a *read* token under Settings, Access Tokens.
2. **Kaggle account.** Verify your phone number (needed for GPUs and internet).
3. **Import this notebook.** Kaggle, Create, New Notebook, File, Import Notebook.
4. **Notebook settings** (right-hand panel): Accelerator **GPU T4 x2**,
   Internet **on**.
5. **Secret.** Add-ons, Secrets, add `HF_TOKEN` with your token, and tick it
   for this notebook.
6. **Shot data.** Add Input, Upload, a new dataset containing `images.csv`.

## Each run

- Leave `RUN_MODE = "check"` the first time. It makes two images and prints the
  real time per image and the projected time for the whole film.
- Then set `RUN_MODE = "full"` and use **Save Version, Save & Run All
  (Commit)**. That keeps running after you close the tab, and the images land in
  the version's Output.
- **Every later run:** Add Input, Notebooks, Your Work, and pick the latest
  version of this notebook. That brings in three things from the earlier run:
  the images already made (skipped, so a cut-off run picks up where it
  stopped), the prompt encodings (no 10 GB text-encoder download unless prompts
  changed), and the saved 4-bit model (no 24 GB download; start-up drops from
  about 20 minutes to a few).
- To redo some images, list them in `ONLY` and set `REDO = True`.

In [ ]:
# ---- settings ---------------------------------------------------------------
import os
from pathlib import Path

RUN_MODE = os.environ.get("RUN_MODE", "check")   # "dry" (no model), "check" (2 images), "full"
MODEL_ID = os.environ.get("MODEL_ID", "black-forest-labs/FLUX.1-schnell")

WIDTH = int(os.environ.get("WIDTH", 1536))       # 16:9, both multiples of 16
HEIGHT = int(os.environ.get("HEIGHT", 864))
STEPS = 4                                        # schnell is made for 4 steps
VARIANTS = 2                                     # versions per image to pick from
SEED_BASE = 20151101                             # change to get a fresh set of variants
MAX_SEQ = 256                                    # prompt token limit for schnell

ONLY = []        # e.g. ["bridge_aerial", "prison_gate"] to work on just these
REDO = False     # True = overwrite images that already exist
SAVE_MODEL = True  # save the 4-bit model to the output (about 7 GB) so later runs skip the download

ON_KAGGLE = Path("/kaggle").exists()
OUT = Path(os.environ.get("OUT", "/kaggle/working/xu-xiang" if ON_KAGGLE else "out"))
INSTALL = os.environ.get("INSTALL", "1") == "1"

In [ ]:
# ---- install (pinned to the versions this notebook was tested with) ----------
import subprocess, sys
if RUN_MODE != "dry" and INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "diffusers==0.43.0", "transformers==5.23.0", "accelerate==1.16.0",
                    "bitsandbytes==0.50.2", "sentencepiece", "protobuf"], check=True)

In [ ]:
# ---- read the image list, work out what still needs making -------------------
import csv, glob, shutil, zlib

def find_images_csv():
    hits = sorted(glob.glob("/kaggle/input/**/images.csv", recursive=True))
    if Path("images.csv").exists():
        hits.append("images.csv")
    if not hits:
        raise FileNotFoundError("images.csv not found. Upload it as a Kaggle dataset (see setup step 6).")
    return hits[0]

csv_path = find_images_csv()
rows = list(csv.DictReader(open(csv_path, encoding="utf-8")))
if ONLY:
    rows = [r for r in rows if r["image_id"] in ONLY]
print(f"{len(rows)} images in {csv_path}")

FRAMES = OUT / "frames"
EMBEDS = OUT / "embeds"
FRAMES.mkdir(parents=True, exist_ok=True)
EMBEDS.mkdir(parents=True, exist_ok=True)

# bring in images and prompt encodings from earlier runs added as inputs
for pattern, folder in [("frames/*.png", FRAMES), ("embeds/*.pt", EMBEDS)]:
    for old in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dest = folder / Path(old).name
        if not dest.exists():
            shutil.copy(old, dest)

# a 4-bit model saved by an earlier run, if one was added as an input
_cache = sorted(glob.glob("/kaggle/input/**/model-cache/transformer/config.json", recursive=True))
MODEL_CACHE = os.environ.get("MODEL_CACHE") or (str(Path(_cache[0]).parent) if _cache else None)
print("Saved model:", MODEL_CACHE or "none, will download")

def seed_for(image_id, variant):
    # depends only on the id, so reordering the CSV never changes an image
    return (SEED_BASE + zlib.crc32(image_id.encode()) * 10 + variant) % 2**31

jobs = []
for r in rows:
    for v in range(1, VARIANTS + 1):
        path = FRAMES / f"{r['image_id']}__v{v}.png"
        if REDO or not path.exists():
            jobs.append(dict(image_id=r["image_id"], variant=v, seed=seed_for(r["image_id"], v),
                             prompt=r["prompt"], path=path))

if RUN_MODE == "check":
    firsts = [j for j in jobs if j["variant"] == 1]
    jobs = firsts[:2] or jobs[:2]

print(f"{len(jobs)} images to make now ({RUN_MODE} mode)")

In [ ]:
# ---- step 1: turn every prompt into text embeddings, once ---------------------
# The text encoders run in full fp32 (the diffusers docs advise this on T4-class
# GPUs, where fp16 text encoding changes the image). They are then unloaded so
# each GPU has room for its own copy of the image model.
import gc, time

def embed_path(job):
    return EMBEDS / f"{job['image_id']}-{zlib.crc32(job['prompt'].encode()):08x}.pt"

HF_TOKEN = os.environ.get("HF_TOKEN")
if ON_KAGGLE and RUN_MODE != "dry":
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN secret found:", e)

if RUN_MODE != "dry":
    import torch
    from diffusers import FluxPipeline
    from transformers import CLIPTextModel, T5EncoderModel

    CUDA = torch.cuda.is_available()
    GPUS = torch.cuda.device_count() if CUDA else 0
    print(f"GPUs: {GPUS}" + (" (" + ", ".join(torch.cuda.get_device_name(i) for i in range(GPUS)) + ")" if GPUS else " - running on CPU"))

    todo = {}
    for j in jobs:
        if not embed_path(j).exists():
            todo[embed_path(j)] = j["prompt"]

    if todo:
        t0 = time.time()
        dev = "cuda:0" if CUDA else "cpu"
        t5_kwargs = dict(device_map="auto") if CUDA else {}
        te2 = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_2", dtype=torch.float32,
                                             token=HF_TOKEN, **t5_kwargs)
        te1 = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", dtype=torch.float32,
                                            token=HF_TOKEN).to(dev)
        encoder = FluxPipeline.from_pretrained(MODEL_ID, transformer=None, vae=None,
                                               text_encoder=te1, text_encoder_2=te2,
                                               dtype=torch.float32, token=HF_TOKEN)
        with torch.no_grad():
            for path, prompt in todo.items():
                pe, pooled, _ = encoder.encode_prompt(prompt=prompt, prompt_2=None, device=dev,
                                                      max_sequence_length=MAX_SEQ)
                torch.save({"prompt_embeds": pe.to(torch.float16).cpu(),
                            "pooled": pooled.to(torch.float16).cpu()}, path)
        del encoder, te1, te2
        gc.collect()
        if CUDA:
            torch.cuda.empty_cache()
        print(f"Encoded {len(todo)} prompts in {time.time() - t0:.0f}s")
    else:
        print("All prompts already encoded")

In [ ]:
# ---- step 2: load one image model per GPU -------------------------------------
# On GPU the transformer is quantised to 4-bit NF4 as it loads (about 7 GB per
# T4 instead of 24 GB). It downloads once; the second GPU reuses the files.
# If an earlier run's saved 4-bit model is an input, it loads from there.
if RUN_MODE != "dry" and jobs:
    from diffusers import (AutoencoderKL, BitsAndBytesConfig, FlowMatchEulerDiscreteScheduler,
                           FluxTransformer2DModel)

    def make_pipe(dev):
        if dev.startswith("cuda"):
            dtype = torch.float16       # T4 has no fast bfloat16
            if MODEL_CACHE:             # already 4-bit; the quantisation settings are saved with it
                transformer = FluxTransformer2DModel.from_pretrained(MODEL_CACHE, dtype=dtype, device_map={"": dev})
            else:
                quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=dtype)
                transformer = FluxTransformer2DModel.from_pretrained(
                    MODEL_ID, subfolder="transformer", quantization_config=quant, dtype=dtype,
                    device_map={"": dev}, token=HF_TOKEN)
        else:
            dtype = torch.float32
            transformer = FluxTransformer2DModel.from_pretrained(
                MODEL_CACHE or MODEL_ID, **({} if MODEL_CACHE else {"subfolder": "transformer", "token": HF_TOKEN}), dtype=dtype)
        vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", dtype=dtype, token=HF_TOKEN).to(dev)
        scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler", token=HF_TOKEN)
        pipe = FluxPipeline(scheduler=scheduler, vae=vae, transformer=transformer,
                            text_encoder=None, tokenizer=None, text_encoder_2=None, tokenizer_2=None)
        pipe.set_progress_bar_config(disable=True)
        return pipe

    devices = [f"cuda:{i}" for i in range(GPUS)] or ["cpu"]
    devices = devices[:max(1, min(len(devices), len(jobs)))]
    pipes = []
    for d in devices:
        t0 = time.time()
        pipes.append(make_pipe(d))
        print(f"Model ready on {d} in {time.time() - t0:.0f}s")
        if len(pipes) == 1 and SAVE_MODEL and not MODEL_CACHE:
            try:
                t0 = time.time()
                pipes[0].transformer.save_pretrained(OUT / "model-cache" / "transformer")
                print(f"Saved the model for later runs in {time.time() - t0:.0f}s")
            except Exception as e:   # never lose a run over the cache
                print("Could not save the model (later runs will download it again):", e)

In [ ]:
# ---- step 3: generate ---------------------------------------------------------
import threading
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw

LOG = OUT / "log.csv"
lock = threading.Lock()
timings = []

def log(job, seconds, device):
    with lock:
        timings.append(seconds)
        new = not LOG.exists()
        with open(LOG, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(["image_id", "variant", "seed", "seconds", "device", "width", "height"])
            w.writerow([job["image_id"], job["variant"], job["seed"], f"{seconds:.1f}", device, WIDTH, HEIGHT])
        print(f"  {job['path'].name}  {seconds:.1f}s on {device}  ({len(timings)}/{len(jobs)})", flush=True)

def placeholder(job):
    img = Image.new("RGB", (WIDTH, HEIGHT), (58, 62, 66))
    d = ImageDraw.Draw(img)
    d.text((24, 24), f"{job['image_id']}  v{job['variant']}  seed {job['seed']}", fill=(240, 236, 228))
    words, line, y = job["prompt"].split(), "", 60
    for w_ in words:
        if len(line) + len(w_) > 110:
            d.text((24, y), line, fill=(180, 176, 168)); y += 18; line = ""
        line += w_ + " "
    d.text((24, y), line, fill=(180, 176, 168))
    return img

def worker(k):
    device = devices[k] if RUN_MODE != "dry" else "none"
    for job in jobs[k::len(devices)] if RUN_MODE != "dry" else jobs:
        t0 = time.time()
        if RUN_MODE == "dry":
            img = placeholder(job)
        else:
            e = torch.load(embed_path(job))
            dt = torch.float16 if device.startswith("cuda") else torch.float32
            img = pipes[k](prompt_embeds=e["prompt_embeds"].to(device, dt),
                           pooled_prompt_embeds=e["pooled"].to(device, dt),
                           width=WIDTH, height=HEIGHT, num_inference_steps=STEPS, guidance_scale=0.0,
                           max_sequence_length=MAX_SEQ,
                           generator=torch.Generator("cpu").manual_seed(job["seed"])).images[0]
        img.save(job["path"])
        log(job, time.time() - t0, device)

wall = time.time()
if jobs:
    if RUN_MODE == "dry":
        worker(0)
    else:
        with ThreadPoolExecutor(len(devices)) as pool:
            list(pool.map(worker, range(len(devices))))
wall = time.time() - wall
print(f"Made {len(jobs)} images in {wall / 60:.1f} min")

In [ ]:
# ---- step 4: contact sheets, review page, picks file -------------------------
from html import escape

all_rows = list(csv.DictReader(open(csv_path, encoding="utf-8")))
SHEETS = OUT / "contact"
SHEETS.mkdir(exist_ok=True)
thumb_w = 640
html = ["<!doctype html><meta charset='utf-8'><title>Xu Xiang frames</title>",
        "<style>body{font:14px system-ui;background:#1d1f21;color:#eee;margin:24px}"
        "img{width:32%;margin-right:1%;border:1px solid #444}h3{margin:28px 0 6px}"
        "p{color:#aaa;max-width:1100px}</style><h1>Xu Xiang frames</h1>"]
made = 0
for r in all_rows:
    files = sorted(FRAMES.glob(f"{r['image_id']}__v*.png"))
    if not files:
        continue
    made += 1
    thumbs = [Image.open(f).convert("RGB") for f in files]
    th = int(thumb_w * thumbs[0].height / thumbs[0].width)
    sheet = Image.new("RGB", (thumb_w * len(thumbs), th + 28), (29, 31, 33))
    d = ImageDraw.Draw(sheet)
    for i, (f, im) in enumerate(zip(files, thumbs)):
        sheet.paste(im.resize((thumb_w, th)), (i * thumb_w, 28))
        d.text((i * thumb_w + 8, 8), f"{r['image_id']}  {f.stem.split('__')[1]}", fill=(238, 238, 238))
    sheet.save(SHEETS / f"{r['image_id']}.jpg", quality=85)
    html.append(f"<h3>{escape(r['image_id'])} <small>(used in {escape(r.get('uses', '?'))} shots)</small></h3>")
    html.append(f"<p>{escape(r['prompt'])}</p>")
    html += [f"<img src='frames/{f.name}' title='{f.stem}'>" for f in files]
(OUT / "review.html").write_text("\n".join(html), encoding="utf-8")

picks = OUT / "picks.csv"
if not picks.exists():
    with open(picks, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image_id", "pick", "note"])
        for r in all_rows:
            w.writerow([r["image_id"], "", ""])
print(f"Contact sheets for {made} images in {SHEETS}; open review.html to choose; record choices in picks.csv")

In [ ]:
# ---- step 5: timing, and what the whole film will take ------------------------
if timings and RUN_MODE != "dry":
    per_image = sum(timings) / len(timings)
    gpus = len(devices)
    total = len(all_rows) * VARIANTS
    print(f"Average {per_image:.1f}s per image on each of {gpus} device(s), {WIDTH}x{HEIGHT}, {STEPS} steps")
    print(f"Whole film: {total} images ({len(all_rows)} x {VARIANTS} variants) "
          f"is about {total * per_image / gpus / 60:.0f} min of generation, "
          f"plus about 15-25 min of downloading and loading at the start")
    if RUN_MODE == "check":
        print("Note: the first image on each GPU includes warm-up, so this estimate runs a little high.")